# 指派问题（Assignment）：一个模型一个 Notebook

## 这个模型解决什么问题
n 个人做 n 件事，一件事只能一个人做，一个人只能做一件事，让总成本最小（或总收益最大）。

```
min  Σ Σ c_ij · x_ij
s.t. Σ_j x_ij = 1   每个任务恰好指派一个人
     Σ_i x_ij = 1   每个人恰好领一个任务
     x_ij ∈ {0,1}
```

## 为什么不用整数规划求解
这是完全二部图的完美匹配，用**匈牙利算法**（`scipy.optimize.linear_sum_assignment`）
多项式时间直接得到最优解，比建 MILP 快得多，也不会有求解器容差问题。
人数和任务数不等时：多的一方会自动"有空闲"，行或列不足部分按 0 补齐即可。

## 两种数据输入方式
| 方式 | 数据长什么样 |
| --- | --- |
| 已处理矩阵 | 一张 n×n 的表，行是人、列是任务、值是成本 |
| 原始表（长表） | 三列：人员 / 任务 / 成本，可能有缺项，需要先 pivot 成矩阵 |

## 目录
0 环境 / 1 配置 / 2 数据接口 / 3 求解 / 4 矩阵示例 / 5 长表示例 / 6 导出

> 依赖 `scipy`；不需要 `pulp` 或 `ortools`。


In [1]:
# -*- coding: utf-8 -*-
# =====================================================================
# 0. 环境与路径（每个 Notebook 都带这一段，保证单独打开就能跑）
# =====================================================================
# 依赖：numpy、pandas、scipy（>=1.9，用到 milp）
# 可选：matplotlib（画图）、openpyxl（导出 xlsx）
# 缺哪个就装：pip install numpy pandas scipy matplotlib openpyxl
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import optimize as sp_opt
from scipy.optimize import Bounds, LinearConstraint

try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 不设中文字体，图里的标题和标签会变成方块
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
    HAS_MPL = True
except ImportError:
    HAS_MPL = False


def find_root(marker="导入数据"):
    """向上逐级找含 导入数据/ 的目录。

    Notebook 放在 优化类模型/ 下也能找到项目根，换电脑、换目录都不用改死路径。
    """
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).is_dir():
            return p
    return Path.cwd()


ROOT = find_root()
DATA_DIR = ROOT / "导入数据"
OUT_DIR = ROOT / "输出数据"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("项目根目录：", ROOT)
print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp_opt.__name__ and __import__("scipy").__version__)


项目根目录： D:\git项目
numpy 2.4.6 | pandas 3.1.0 | scipy 1.17.1


In [2]:
# =====================================================================
# 1. 配置区
# =====================================================================
CFG = dict(
    maximize=False,     # False=最小化总成本；True=最大化总收益
    tag="指派方案",
    make_plots=True,
    save_excel=True,
)
print("配置：", CFG)


配置： {'maximize': False, 'tag': '指派方案', 'make_plots': True, 'save_excel': True}


## 2. 数据接口


In [3]:
# =====================================================================
# 2. 数据接口：读文件 → 清洗 → 变成模型能吃的表
# =====================================================================
# 这一格是「原始数据直接放进来」的关键：不用你手动去空格、转数字、去重。
# 如果数据已经处理好，可以跳过清洗，直接用后面的 load_matrix 读矩阵。
#
# ⚠️ 999 是否算缺失取决于题目：若它是真实观测值，把它从 NA_MARKERS 里删掉。
NA_MARKERS = ("", "NA", "N/A", "na", "缺测", "未填", "无", "999", 999, "9999", 9999)


def load_table(data, sheet_name=0, encoding="utf-8-sig"):
    """读 csv / xlsx，也接受现成的 DataFrame。

    - csv 先按 utf-8-sig 读（吃掉 Excel 导出的 BOM），失败再退回 gbk
    - 列名首尾空格自动去掉（" 单耗" → "单耗"）
    """
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        p = Path(data)
        if not p.exists():
            raise FileNotFoundError(f"找不到数据文件：{p}")
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p, sheet_name=sheet_name)
        else:
            try:
                df = pd.read_csv(p, encoding=encoding)
            except UnicodeDecodeError:
                df = pd.read_csv(p, encoding="gbk")
    df.columns = [str(c).strip() for c in df.columns]
    return df


def clean_frame(df, na_markers=NA_MARKERS, drop_missing_rate=50.0,
                fill_num="median", diagnostics=None):
    """把原始附件洗成能建模的表，并把每一步记进 diagnostics。

    顺序有因果，别随意调换：
    ① 哨兵值 → NaN       999 / 缺测 / 未填 统一当缺失
    ② 文本列去空格       否则 "东部" 与 " 东部" 会被当成两个类别
    ③ 文本数字转数值     " 12.5 " 这种带空格的数字
    ④ 删完全重复行
    ⑤ 高缺失列删列       删列不删行，保住样本量
    ⑥ 缺失填补           数值列中位数/均值，文本列众数

    参数
    ----
    diagnostics : 传一个 list 进来，会往里追加每一步的说明，最后可以导出给论文用
    """
    diag = diagnostics if diagnostics is not None else []
    d = df.copy()
    d.columns = [str(c).strip() for c in d.columns]

    before = int(d.isna().sum().sum())
    d = d.replace(list(na_markers), np.nan)
    diag.append({"步骤": "哨兵值转缺失",
                 "说明": f"新增缺失 {int(d.isna().sum().sum()) - before} 个"})

    obj_cols = [c for c in d.columns
                if pd.api.types.is_string_dtype(d[c]) or pd.api.types.is_object_dtype(d[c])]
    for c in obj_cols:
        d[c] = d[c].astype("string").str.strip()
    if obj_cols:
        diag.append({"步骤": "文本列去空格", "说明": f"处理 {len(obj_cols)} 列"})

    num_cols, cat_cols = [], []
    for c in d.columns:
        s = d[c]
        if pd.api.types.is_numeric_dtype(s):
            num_cols.append(c)
            continue
        conv = pd.to_numeric(s.astype(str).str.strip(), errors="coerce")
        if conv.notna().sum() / max(s.notna().sum(), 1) > 0.9:
            d[c] = conv
            num_cols.append(c)
        else:
            cat_cols.append(c)
    diag.append({"步骤": "列类型识别",
                 "说明": f"数值列 {len(num_cols)} 个、类别列 {len(cat_cols)} 个"})

    n_dup = int(d.duplicated().sum())
    d = d.drop_duplicates(keep="first").reset_index(drop=True)
    diag.append({"步骤": "去完全重复", "说明": f"删除 {n_dup} 行"})

    rate = d.isna().mean() * 100
    drop = list(rate[rate >= drop_missing_rate].index)
    if drop:
        d = d.drop(columns=drop)
        diag.append({"步骤": "删除高缺失列",
                     "说明": f"删除 {drop}（缺失率 ≥ {drop_missing_rate}%），删列不删行"})

    filled = []
    for c in num_cols:
        if c in d.columns and d[c].isna().any():
            fill = {"median": d[c].median(), "mean": d[c].mean(), "zero": 0}[fill_num]
            d[c] = d[c].fillna(fill)
            filled.append(c)
    for c in cat_cols:
        if c in d.columns and d[c].isna().any():
            m = d[c].mode(dropna=True)
            d[c] = d[c].fillna(m.iat[0] if len(m) else "未知")
            filled.append(c)
    if filled:
        diag.append({"步骤": "缺失填补", "说明": f"填补 {filled}，数值方式 {fill_num}"})

    diag.append({"步骤": "清洗完成", "说明": f"{d.shape[0]} 行 × {d.shape[1]} 列"})
    return d


def pivot_long(df, index, columns, values, agg="sum"):
    """长表转宽表：产品-资源-单耗 三列 → 产品 × 资源 矩阵。

    真实附件常见这种长表；同一 (产品, 资源) 出现多次时按 agg 聚合。
    """
    wide = df.pivot_table(index=index, columns=columns, values=values, aggfunc=agg).reset_index()
    wide.columns = [str(c) for c in wide.columns]
    return wide


def load_matrix(data, sheet_name=0):
    """读成本矩阵 / 距离矩阵：第一列当行名，其余列转成数值。"""
    df = load_table(data, sheet_name)
    if df.shape[1] > 1 and not pd.api.types.is_numeric_dtype(df.iloc[:, 0]):
        df = df.set_index(df.columns[0])
    df.index = [str(i) for i in df.index]
    df.columns = [str(c) for c in df.columns]
    return df.astype(float)


## 3. 求解：匈牙利算法

不用建 MILP，`linear_sum_assignment` 直接给最优匹配。


In [4]:
# =====================================================================
# 3. 求解：匈牙利算法
# =====================================================================
def solve_assignment(cost, maximize=False, row_names=None, col_names=None):
    """指派问题求解。

    cost : 二维数组或 DataFrame（行=人，列=任务）
    返回 dict：配对表 / 总成本 / 是否最优 / 用时。
    """
    t0 = time.time()
    if isinstance(cost, pd.DataFrame):
        row_names = list(cost.index) if row_names is None else row_names
        col_names = list(cost.columns) if col_names is None else col_names
        M = cost.to_numpy(dtype=float)
    else:
        M = np.atleast_2d(np.asarray(cost, dtype=float))
        row_names = row_names or [f"人员{i+1}" for i in range(M.shape[0])]
        col_names = col_names or [f"任务{j+1}" for j in range(M.shape[1])]

    # linear_sum_assignment 返回每行被指派到的列号
    r, c = sp_opt.linear_sum_assignment(M, maximize=maximize)
    pairs = pd.DataFrame({
        "人员": [row_names[i] for i in r],
        "任务": [col_names[j] for j in c],
        "成本": np.round(M[r, c], 4),
    })
    total = float(M[r, c].sum())
    return dict(pairs=pairs, total=total, matrix=M,
                row_names=row_names, col_names=col_names,
                maximize=maximize, elapsed=time.time() - t0)


def assignment_summary(res):
    """摘要表 + 检查是否每行每列各用一次。"""
    t = pd.DataFrame([{
        "总成本": round(res["total"], 4),
        "方向": "最大化" if res["maximize"] else "最小化",
        "人数": len(res["row_names"]), "任务数": len(res["col_names"]),
        "每人一项": res["pairs"]["人员"].is_unique,
        "每任务一人": res["pairs"]["任务"].is_unique,
        "用时(秒)": round(res["elapsed"], 4),
    }])
    return t


## 4. 示例 A：已处理矩阵


In [5]:
# =====================================================================
# 4. 示例 A：已处理矩阵（最常见）
# =====================================================================
# 一张 4×4 的成本表：行是人员，列是任务，格子里是"派他做这个任务要花多少"。
cost_matrix = pd.DataFrame(
    [[9, 2, 7, 8],
     [6, 4, 3, 7],
     [5, 8, 1, 8],
     [7, 6, 9, 4]],
    index=["甲", "乙", "丙", "丁"],
    columns=["任务1", "任务2", "任务3", "任务4"])
print("成本矩阵：")
print(cost_matrix.to_string())

res_a = solve_assignment(cost_matrix, maximize=CFG["maximize"])
print("\n=== 最优指派 ===")
print(res_a["pairs"].to_string(index=False))
print("\n", assignment_summary(res_a).to_string(index=False))
print("\n核对：总成本 =", res_a["total"],
      "| 手算 =", "、".join(f"{a}-{b}({v})" for a, b, v in
                            zip(res_a["pairs"]["人员"], res_a["pairs"]["任务"],
                                res_a["pairs"]["成本"])))


成本矩阵：
   任务1  任务2  任务3  任务4
甲    9    2    7    8
乙    6    4    3    7
丙    5    8    1    8
丁    7    6    9    4

=== 最优指派 ===
人员  任务  成本
 甲 任务2 2.0
 乙 任务1 6.0
 丙 任务3 1.0
 丁 任务4 4.0

  总成本  方向  人数  任务数  每人一项  每任务一人  用时(秒)
13.0 最小化   4    4  True   True    0.0

核对：总成本 = 13.0 | 手算 = 甲-任务2(2.0)、乙-任务1(6.0)、丙-任务3(1.0)、丁-任务4(4.0)


## 5. 示例 B：长表 → 矩阵


In [6]:
# =====================================================================
# 5. 示例 B：长表 → 矩阵（原始数据模式）
# =====================================================================
# 真实附件常常是"人员-任务-成本"三列长表，还可能有缺项。
# 先 pivot 成矩阵，再套同一套求解函数。
long_table = pd.DataFrame({
    "人员": ["甲", "甲", "甲", "甲", "乙", "乙", "乙", "乙",
             "丙", "丙", "丙", "丙", "丁", "丁", "丁", "丁"],
    "任务": ["任务1", "任务2", "任务3", "任务4"] * 4,
    "成本": [9, 2, 7, 8, 6, 4, 3, 7, 5, 8, 1, 8, 7, 6, 9, 4],
})
# 故意删掉一条，模拟附件里缺项
long_table = long_table.drop(index=5)
print("原始长表（有缺项）：", long_table.shape)
print(long_table.head(8).to_string(index=False))

wide = pivot_long(long_table, "人员", "任务", "成本", agg="sum")
wide = wide.set_index("人员")
# 缺项用该任务的平均成本补（比赛里也可以选择丢弃该组合，但要说明理由）
wide = wide.fillna(wide.mean())
print("\n转成矩阵并补缺：")
print(wide.round(2).to_string())

res_b = solve_assignment(wide, maximize=CFG["maximize"])
print("\n=== 最优指派（长表模式）===")
print(res_b["pairs"].to_string(index=False))
print("\n", assignment_summary(res_b).to_string(index=False))


原始长表（有缺项）： (15, 3)
人员  任务  成本
 甲 任务1   9
 甲 任务2   2
 甲 任务3   7
 甲 任务4   8
 乙 任务1   6
 乙 任务3   3
 乙 任务4   7
 丙 任务1   5

转成矩阵并补缺：
    任务1   任务2  任务3  任务4
人员                     
丁   7.0  6.00  9.0  4.0
丙   5.0  8.00  1.0  8.0
乙   6.0  5.33  3.0  7.0
甲   9.0  2.00  7.0  8.0

=== 最优指派（长表模式）===
人员  任务  成本
 丁 任务4 4.0
 丙 任务3 1.0
 乙 任务1 6.0
 甲 任务2 2.0

  总成本  方向  人数  任务数  每人一项  每任务一人  用时(秒)
13.0 最小化   4    4  True   True    0.0


## 6. 结果导出


In [7]:
# =====================================================================
# 结果导出：CSV + 多 Sheet Excel + 图
# =====================================================================
def export_result(tables, cfg, fig=None, fig_name="结果图"):
    """把结果写成文件，返回写出的路径。

    tables : {Sheet 名: DataFrame}，第一张建议放"摘要"
    cfg    : 配置字典，用 tag / make_plots / save_excel 三个键
    fig    : 传了 matplotlib 图对象就保存；cfg["make_plots"]=False 时连图都不生成
    """
    d = OUT_DIR / f"优化_{cfg['tag']}"
    d.mkdir(parents=True, exist_ok=True)
    written = {}

    first = next(iter(tables.values()))
    p_csv = d / f"{cfg['tag']}_主表.csv"
    first.to_csv(p_csv, index=False, encoding="utf-8-sig")
    written["主表"] = p_csv

    if cfg.get("save_excel", True):
        p_xlsx = d / f"{cfg['tag']}_结果汇总.xlsx"
        with pd.ExcelWriter(p_xlsx, engine="openpyxl") as w:
            for name, frame in tables.items():
                frame.to_excel(w, sheet_name=str(name)[:28], index=False)
        written["汇总"] = p_xlsx
        print(f"[表格] {p_xlsx.name}（{len(tables)} 个 Sheet）")

    if fig is not None and cfg.get("make_plots", True) and HAS_MPL:
        p_png = d / f"{cfg['tag']}_{fig_name}.png"
        fig.savefig(p_png, dpi=150, bbox_inches="tight")
        if "agg" not in matplotlib.get_backend().lower():   # 无界面后端不弹窗，也不报警告
            plt.show()
        plt.close(fig)          # 关掉画布：反复运行不会在内存里堆图
        written["图片"] = p_png
        print(f"[图片] {p_png.name}")

    print("[输出] ", d)
    return written


In [8]:
# =====================================================================
# 6. 结果导出与热力图
# =====================================================================
fig = None
if CFG["make_plots"] and HAS_MPL:
    M = res_a["matrix"]
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    im = ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(res_a["col_names"], rotation=25, ha="right")
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(res_a["row_names"])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center",
                    color="#333", fontsize=10)
    # 把选中的格子圈出来
    for _, row in res_a["pairs"].iterrows():
        i = res_a["row_names"].index(row["人员"]); j = res_a["col_names"].index(row["任务"])
        ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, fill=False,
                                   edgecolor="#d95f5f", lw=2.2))
    ax.set_title(f"指派结果（总成本 {res_a['total']:.0f}，红框=最优配对）")
    fig.colorbar(im, ax=ax, shrink=.8)
    fig.tight_layout()

written = export_result(
    {"结果摘要": assignment_summary(res_a), "指派方案": res_a["pairs"],
     "成本矩阵": pd.DataFrame(res_a["matrix"], index=res_a["row_names"],
                          columns=res_a["col_names"]).reset_index()},
    CFG, fig=fig, fig_name="指派热力图")
print(written)


[表格] 指派方案_结果汇总.xlsx（3 个 Sheet）
[图片] 指派方案_指派热力图.png
[输出]  D:\git项目\输出数据\优化_指派方案
{'主表': WindowsPath('D:/git项目/输出数据/优化_指派方案/指派方案_主表.csv'), '汇总': WindowsPath('D:/git项目/输出数据/优化_指派方案/指派方案_结果汇总.xlsx'), '图片': WindowsPath('D:/git项目/输出数据/优化_指派方案/指派方案_指派热力图.png')}


换题目时按三步走：

1. 数据放进 `导入数据/`（csv / xlsx 都行）；
2. 改本 Notebook 第 1 个代码单元里的 `CFG`：文件名、列名映射、方向、输出名；
3. `Run All`，结果写到 `输出数据/优化_{tag}/`，同名覆盖，不会越跑越多。

**两种输入方式怎么选**

- 数据还脏（有空行、重复、999、文本数字、列名带空格）→ 走「原始数据」示例，让 `clean_frame` 处理；
- 数据已经清洗/标准化完（比如上一节 `数据学习.ipynb` 输出的 `final_cleaned.csv`）→ 走「已处理数据」示例，直接给矩阵或宽表。

不要混：**同一次建模只用一种入口**，否则等于把同一份数据预处理了两遍。
